# Trabajo Final de Inteligencia Artificial
## Avance 1 — Selección del tema y Análisis Exploratorio de Datos (EDA)

**Tema:** Clasificación de la severidad del Dengue en el Perú (2000–2024)

**Dataset:** `Dengue/datos_abiertos_vigilancia_dengue_2000_2024.csv`
(vigilancia epidemiológica, MINSA — datosabiertos.gob.pe)

En este avance se cubre:
1. Justificación del tema
2. Descripción del dataset (diccionario de variables)
3. Carga y limpieza de datos
4. Análisis exploratorio de datos (EDA)
5. Definición del problema


## 0. Librerías

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)
print('pandas', pd.__version__)

pandas 3.0.3


## 1. Justificación del tema

El **dengue** es una enfermedad viral transmitida por el mosquito *Aedes aegypti*
y constituye un problema de salud pública recurrente en el Perú, con brotes
epidémicos cada año. Predecir la **severidad** de un caso (sin signos de alarma,
con signos de alarma o grave) a partir de variables demográficas, temporales y
geográficas tiene valor para la priorización de atención y la vigilancia epidemiológica.

**Cumplimiento de requisitos del trabajo:**
- ✅ **≥ 10 mil registros**: el dataset tiene más de 1 millón de registros.
- ✅ **≥ 10 características interesantes**: a partir de las variables base se construyen
  (en el Avance 2) características temporales, demográficas y epidemiológicas, evitando
  variables inválidas (nombre, dirección, ubigeo/códigos).


In [2]:
# Carga del dataset (separador ';' y manejo del BOM en la cabecera)
RUTA = '../data/raw/datos_abiertos_vigilancia_dengue_2000_2024.csv'
df = pd.read_csv(RUTA, sep=';', encoding='utf-8-sig', dtype=str)
print('Dimensiones (filas, columnas):', df.shape)
df.head()

Dimensiones (filas, columnas): (1029421, 14)


,departamento,provincia,distrito,localidad,enfermedad,ano,semana,diagnostic,diresa,ubigeo,localcod,edad,tipo_edad,sexo
0,LORETO,ALTO AMAZONAS,YURIMAGUAS,000101,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,NaN,58,A,M
1,LORETO,ALTO AMAZONAS,YURIMAGUAS,000101,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,NaN,44,A,F
2,LORETO,ALTO AMAZONAS,YURIMAGUAS,BARRIO MORALILLOS,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,NaN,21,A,F
3,LORETO,MAYNAS,IQUITOS,005902,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160101,NaN,74,A,M
4,LORETO,MAYNAS,IQUITOS,CARDOZO,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160101,NaN,42,A,F


In [3]:
# Verificación del requisito de tamaño
print(f'Número de registros: {len(df):,}')
print('¿Cumple ≥ 10 000 registros?', len(df) >= 10_000)

Número de registros: 1,029,421
¿Cumple ≥ 10 000 registros? True


## 2. Descripción del dataset (diccionario de variables)

| Variable | Descripción | ¿Válida para el modelo? |
|---|---|---|
| `departamento`, `provincia`, `distrito`, `localidad` | Ubicación geográfica del caso | ❌ Geográficas (tipo dirección) |
| `enfermedad` | Clasificación clínica del dengue (texto) | 🎯 **Variable objetivo** |
| `diagnostic` | Código CIE-10 del diagnóstico (A97.0/.1/.2) | 🎯 Equivalente al objetivo |
| `ano` | Año de notificación | ✅ Temporal |
| `semana` | Semana epidemiológica (1–53) | ✅ Temporal |
| `diresa`, `ubigeo`, `localcod` | Códigos administrativos/geográficos | ❌ Identificadores |
| `edad` | Edad (en la unidad de `tipo_edad`) | ✅ Demográfica |
| `tipo_edad` | Unidad de la edad: A=años, M=meses, D=días | ✅ Auxiliar para normalizar edad |
| `sexo` | Sexo del paciente (M/F) | ✅ Demográfica |

La variable objetivo `enfermedad` se corresponde 1 a 1 con el código `diagnostic`:
- `A97.0` → DENGUE SIN SIGNOS DE ALARMA
- `A97.1` → DENGUE CON SIGNOS DE ALARMA
- `A97.2` → DENGUE GRAVE


In [4]:
# Estructura y tipos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1029421 entries, 0 to 1029420
Data columns (total 14 columns):
 #   Column        Non-Null Count    Dtype
---  ------        --------------    -----
 0   departamento  1029421 non-null  str  
 1   provincia     1029421 non-null  str  
 2   distrito      1029421 non-null  str  
 3   localidad     906823 non-null   str  
 4   enfermedad    1029421 non-null  str  
 5   ano           1029421 non-null  str  
 6   semana        1029421 non-null  str  
 7   diagnostic    1029421 non-null  str  
 8   diresa        1029398 non-null  str  
 9   ubigeo        1029421 non-null  str  
 10  localcod      880780 non-null   str  
 11  edad          1029421 non-null  str  
 12  tipo_edad     1029421 non-null  str  
 13  sexo          1029421 non-null  str  
dtypes: str(14)
memory usage: 110.0 MB


In [5]:
# Valores únicos por columna (para entender la cardinalidad)
df.nunique().sort_values(ascending=False)

localidad       10131
localcod         9777
ubigeo            662
distrito          625
provincia         124
edad              123
semana             53
diresa             32
ano                25
departamento       23
enfermedad          3
diagnostic          3
tipo_edad           3
sexo                2
dtype: int64

## 3. Carga y limpieza de datos

Pasos:
1. Revisar valores nulos.
2. Convertir tipos numéricos (`ano`, `semana`, `edad`).
3. **Normalizar la edad a años** usando `tipo_edad` (M=meses/12, D=días/365).
4. Filtrar edades imposibles (el dato crudo tiene valores basura, p. ej. millones).
5. Normalizar texto (`sexo`, `enfermedad`).
6. Crear la variable objetivo ordenada por severidad.


In [6]:
# 3.1 Valores nulos por columna
nulos = df.isna().sum()
nulos[nulos > 0]

localidad    122598
diresa           23
localcod     148641
dtype: int64

In [7]:
# 3.2 Conversión a numérico
for col in ['ano', 'semana', 'edad']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Normalizar texto
df['sexo'] = df['sexo'].str.strip().str.upper()
df['tipo_edad'] = df['tipo_edad'].str.strip().str.upper()
df['enfermedad'] = df['enfermedad'].str.strip().str.upper()

df[['ano', 'semana', 'edad', 'sexo', 'tipo_edad']].describe(include='all')

,ano,semana,edad,sexo,tipo_edad
count,1.029421e+06,1.029421e+06,1.029421e+06,1029421,1029421
unique,NaN,NaN,NaN,2,3
top,NaN,NaN,NaN,F,A
freq,NaN,NaN,NaN,553677,1022720
mean,2.019257e+03,2.061159e+01,1.476117e+02,NaN,NaN
std,6.133137e+00,1.255885e+01,8.219876e+04,NaN,NaN
min,2.000000e+03,1.000000e+00,0.000000e+00,NaN,NaN
25%,2.017000e+03,1.200000e+01,1.500000e+01,NaN,NaN
50%,2.023000e+03,1.800000e+01,2.800000e+01,NaN,NaN
75%,2.024000e+03,2.500000e+01,4.300000e+01,NaN,NaN


In [8]:
# 3.3 Normalizar la edad a AÑOS según tipo_edad
factor = {'A': 1.0, 'M': 1/12, 'D': 1/365}
df['edad_anios'] = df['edad'] * df['tipo_edad'].map(factor)

print('Antes de filtrar:')
print(df['edad_anios'].describe())

Antes de filtrar:
count    1.029421e+06
mean     1.475669e+02
std      8.219876e+04
min      0.000000e+00
25%      1.500000e+01
50%      2.800000e+01
75%      4.300000e+01
max      7.196364e+07
Name: edad_anios, dtype: float64


In [9]:
# 3.4 Filtrar edades imposibles (rango plausible 0–110 años)
n_antes = len(df)
df = df[(df['edad_anios'] >= 0) & (df['edad_anios'] <= 110)].copy()
print(f'Registros eliminados por edad inválida: {n_antes - len(df):,}')
print(f'Registros restantes: {len(df):,}')
df['edad_anios'].describe()

Registros eliminados por edad inválida: 14
Registros restantes: 1,029,407


count    1.029407e+06
mean     3.059700e+01
std      1.899626e+01
min      0.000000e+00
25%      1.500000e+01
50%      2.800000e+01
75%      4.300000e+01
max      1.100000e+02
Name: edad_anios, dtype: float64

In [10]:
# 3.5 Variable objetivo: severidad ordenada (0 < 1 < 2)
orden_sev = {
    'DENGUE SIN SIGNOS DE ALARMA': 'Sin signos',
    'DENGUE CON SIGNOS DE ALARMA': 'Con signos',
    'DENGUE GRAVE': 'Grave',
}
df['severidad'] = df['enfermedad'].map(orden_sev)
cat_sev = pd.CategoricalDtype(['Sin signos', 'Con signos', 'Grave'], ordered=True)
df['severidad'] = df['severidad'].astype(cat_sev)

print(df['severidad'].value_counts(dropna=False))

severidad
Sin signos    915232
Con signos    110163
Grave           4012
Name: count, dtype: int64


In [11]:
# 3.6 Limpiar filtrando sexo válido y registros sin severidad
df = df[df['sexo'].isin(['M', 'F'])].copy()
df = df.dropna(subset=['severidad', 'ano', 'semana']).copy()
print('Dimensiones finales tras limpieza:', df.shape)

Dimensiones finales tras limpieza: (1029407, 16)


## 4. Análisis Exploratorio de Datos (EDA)

### 4.1 Distribución de la variable objetivo (severidad)

In [12]:
conteo = df['severidad'].value_counts().reindex(['Sin signos', 'Con signos', 'Grave'])
porc = (conteo / len(df) * 100).round(2)
resumen = pd.DataFrame({'casos': conteo, 'porcentaje_%': porc})
print(resumen)

ax = conteo.plot(kind='bar', color=['#2ca02c', '#ff7f0e', '#d62728'])
ax.set_title('Distribución de la severidad del dengue')
ax.set_xlabel('Severidad'); ax.set_ylabel('Número de casos')
plt.xticks(rotation=0)
for i, v in enumerate(conteo):
    ax.text(i, v, f'{v:,}', ha='center', va='bottom')
plt.tight_layout(); plt.show()

             casos  porcentaje_%
severidad                       
Sin signos  915232         88.91
Con signos  110163         10.70
Grave         4012          0.39


C:\Users\danie\AppData\Local\Temp\ipykernel_13192\3373892583.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Se observa un **fuerte desbalance de clases**: la clase *Sin signos* concentra
la gran mayoría de los casos, mientras que *Grave* es muy minoritaria. Esto deberá
tratarse en el Avance 2 (class weights / remuestreo / SMOTE).

### 4.2 Distribución de la edad

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['edad_anios'], bins=50, kde=True, ax=axes[0], color='#1f77b4')
axes[0].set_title('Distribución de la edad (años)')
axes[0].set_xlabel('Edad (años)')

sns.boxplot(data=df, x='severidad', y='edad_anios', ax=axes[1],
            order=['Sin signos', 'Con signos', 'Grave'])
axes[1].set_title('Edad por nivel de severidad')
axes[1].set_xlabel('Severidad'); axes[1].set_ylabel('Edad (años)')
plt.tight_layout(); plt.show()

print(df.groupby('severidad', observed=True)['edad_anios'].describe())

               count       mean        std      min   25%   50%   75%    max
severidad                                                                   
Sin signos  915232.0  30.731169  18.912099  0.00000  15.0  28.0  43.0  110.0
Con signos  110163.0  29.243309  19.367946  0.00274  14.0  26.0  41.0  104.0
Grave         4012.0  37.158717  24.492298  0.00274  17.0  32.0  55.0   99.0


C:\Users\danie\AppData\Local\Temp\ipykernel_13192\3641507838.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 4.3 Distribución por sexo

In [14]:
tab = pd.crosstab(df['sexo'], df['severidad'], normalize='index') * 100
print(tab.round(2))

ct = pd.crosstab(df['sexo'], df['severidad'])
ct = ct[['Sin signos', 'Con signos', 'Grave']]
ct.plot(kind='bar', stacked=True, color=['#2ca02c', '#ff7f0e', '#d62728'])
plt.title('Severidad por sexo'); plt.xlabel('Sexo'); plt.ylabel('Casos')
plt.xticks(rotation=0); plt.legend(title='Severidad')
plt.tight_layout(); plt.show()

severidad  Sin signos  Con signos  Grave
sexo                                    
F               88.25       11.36   0.39
M               89.68        9.94   0.39


C:\Users\danie\AppData\Local\Temp\ipykernel_13192\978902590.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 4.4 Tendencia anual de casos

In [15]:
casos_anio = df.groupby('ano').size()
ax = casos_anio.plot(marker='o', color='#1f77b4')
ax.set_title('Casos de dengue por año (2000–2024)')
ax.set_xlabel('Año'); ax.set_ylabel('Número de casos')
plt.tight_layout(); plt.show()

print('Años con más casos:')
print(casos_anio.sort_values(ascending=False).head())

Años con más casos:
ano
2024    271531
2023    256639
2017     68279
2022     63216
2020     47932
dtype: int64


C:\Users\danie\AppData\Local\Temp\ipykernel_13192\3652177937.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 4.5 Estacionalidad (semana epidemiológica)

In [16]:
casos_sem = df.groupby('semana').size()
ax = casos_sem.plot(color='#9467bd')
ax.set_title('Casos por semana epidemiológica (acumulado 2000–2024)')
ax.set_xlabel('Semana epidemiológica (1–53)'); ax.set_ylabel('Número de casos')
plt.tight_layout(); plt.show()

C:\Users\danie\AppData\Local\Temp\ipykernel_13192\3435380259.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 4.6 Casos por departamento

In [17]:
casos_dep = df.groupby('departamento').size().sort_values(ascending=False).head(15)
ax = casos_dep.plot(kind='barh', color='#17becf')
ax.set_title('Top 15 departamentos por número de casos')
ax.set_xlabel('Número de casos'); ax.invert_yaxis()
plt.tight_layout(); plt.show()

C:\Users\danie\AppData\Local\Temp\ipykernel_13192\1216704139.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 4.7 Resumen del desbalance de clases

In [18]:
prop = df['severidad'].value_counts(normalize=True).reindex(
    ['Sin signos', 'Con signos', 'Grave']) * 100
print('Proporción de clases (%):')
print(prop.round(2))
print(f'\nClase mayoritaria: {prop.idxmax()} ({prop.max():.2f}%)')
print(f'Clase minoritaria: {prop.idxmin()} ({prop.min():.2f}%)')

Proporción de clases (%):
severidad
Sin signos    88.91
Con signos    10.70
Grave          0.39
Name: proportion, dtype: float64

Clase mayoritaria: Sin signos (88.91%)
Clase minoritaria: Grave (0.39%)


## 5. Definición del problema

- **Tipo de problema:** clasificación supervisada **multiclase**.
- **Variable objetivo:** `severidad` con 3 categorías ordinales
  (*Sin signos* < *Con signos* < *Grave*).
- **Variables predictoras (base):** edad (normalizada a años), sexo, año,
  semana epidemiológica y región. En el **Avance 2** se aplicará *feature
  engineering* para alcanzar ≥10 características (codificación cíclica de la
  semana, estación del año, grupos de edad, incidencia/lags por región, etc.).
- **Reto principal:** fuerte **desbalance de clases**, que se abordará con
  técnicas de remuestreo y/o ponderación de clases, y métricas adecuadas
  (F1 macro, ROC-AUC por clase) en lugar de solo *accuracy*.

### Conclusiones del Avance 1
- El dataset cumple holgadamente el requisito de tamaño (>1M de registros).
- La limpieza clave fue la **normalización de la edad** y el filtrado de valores
  imposibles, además de unificar la variable objetivo.
- El EDA confirma desbalance de clases, un patrón **estacional** marcado y la
  concentración de casos en ciertos años (brotes) y departamentos.
- Queda definido un problema de **clasificación multiclase** listo para la etapa
  de modelado del Avance 2.


In [19]:
# Guardar el dataset limpio para reutilizarlo en el Avance 2
cols_finales = ['ano', 'semana', 'edad_anios', 'sexo', 'departamento',
                'provincia', 'enfermedad', 'diagnostic', 'severidad']
df[cols_finales].to_csv('../data/processed/dengue_limpio_avance1.csv', index=False)
print('Dataset limpio guardado en ../data/processed/dengue_limpio_avance1.csv')
print('Dimensiones:', df[cols_finales].shape)

Dataset limpio guardado en ../data/processed/dengue_limpio_avance1.csv
Dimensiones: (1029407, 9)
